In [1]:
from langchain_core.tools import tool 
from langchain_aws import ChatBedrockConverse 
from langchain.agents import create_agent 
from langchain.agents.middleware import SummarizationMiddleware 
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
@tool
def check_course_availability(course_name:str): 
    """Checks if seats are available for a specific training course.""" 
    courses = {
        "Data Science": "5 seats left", 
        "Web Development": "Waiting list", 
        "AI Ethics":"12 seats available"
    } 
    return courses.get(course_name, "Course not found. Please ask about Data Science or Web Dev.") 

@tool
def calculate_emi_plan(total_fee: float, months: int): 
    """Calculates emi payment for a student fee plan""" 
    if months <= 0: 
        return "Duration must be at least 1 month." 
    monthly = total_fee / months 
    return f"The installment plan is {months} payment of ${monthly:.2f} per month."

In [3]:
tools = [check_course_availability, calculate_emi_plan]

In [4]:
llm = ChatBedrockConverse(model_id = "cohere.command-r-plus-v1:0", temperature = 0.4, region_name = 'us-east-1')

In [5]:
summary_llm = ChatBedrockConverse(model_id = 'amazon.nova-lite-v1:0', region_name = 'us-east-1')

In [6]:
SYSTEM_PROMPT = """You are a professional receptionist for  'FutureTech Institute'. 
Your goal is to help prospective students with course info and fee queries. 
Be polite, helpful and concise."""

In [7]:
agent = create_agent(model = llm, tools = tools, system_prompt=SYSTEM_PROMPT, 
                     checkpointer = InMemorySaver(), 
                     middleware = [
                         SummarizationMiddleware(model=summary_llm, trigger = ('tokens', 250), 
                                                 keep = ('messages', 3))])

In [9]:
config = {'configurable':{'thread_id':'stu_001'}} 
while True: 
    query = input("User_query: ") 
    if query == 'exit': 
        break 
    response = agent.invoke(
        {'messages':query}, config = config) 
    print(response['messages'][-1].content)

User_query:  Hi


Hello, how can I help?


User_query:  Could you help me in letting me know about the availability of seats for AI Ethics


There are 12 seats available for the AI Ethics course.


User_query:  great, and for the other courses as well


...

User_query:  how about for 9 months?


For the AI Ethics course, the EMI plan over 9 months would be 9 payments of $1500 per month. For the Data Science course, the EMI plan over 9 months would be 9 payments of $944.44 per month.


User_query:  great, then I will go with Data science


Great, I will go ahead and enrol you in the Data Science course.


User_query:  exit


In [10]:
print(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to inquire about the availability of seats for the AI ethics course and potentially other courses, as well as the cost of the courses and the availability of EMI plans.\n\n## SUMMARY\nThe user asked about the availability of seats for the AI ethics course and other courses. The AI found 12 seats available for the AI ethics course and 5 for the Data Science course, but no information was found about the Web Dev course. The user also inquired about the costs of the AI ethics and Data Science courses and asked if there are any EMI plans available. The AI confirmed the availability of EMI plans for both courses and calculated the monthly installments.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe AI should calculate the EMI plan for the AI Ethics and Data Science courses if the payment period is changed to 9 months.", additional_kwargs={'lc_source': 'summarizati